# Silver -> Gold (Dia 2)

Modelagem dimensional final (esquema estrela) para consumo em Direct Lake.

Saida: `dim_customer`, `dim_seller`, `dim_product`, `dim_date`, `fact_orders`, `fact_order_items` (Delta, com V-Order/OPTIMIZE).

In [ ]:
from pyspark.sql import functions as F

# Lakehouse com esquema (schema-enabled): tabelas sob dbo.
# Se for um Lakehouse legado (sem esquema), deixe SCHEMA = "".
SCHEMA = "dbo"

def tbl(name: str) -> str:
    """Qualifica o nome da tabela com o esquema, quando houver."""
    return f"{SCHEMA}.{name}" if SCHEMA else name

orders = spark.read.table(tbl("slv_orders"))
items = spark.read.table(tbl("slv_order_items"))
payments = spark.read.table(tbl("slv_order_payments"))
customers = spark.read.table(tbl("slv_customers"))
sellers = spark.read.table(tbl("slv_sellers"))
products = spark.read.table(tbl("slv_products"))

In [ ]:
# Dimensoes
dim_customer = customers.select("customer_id", "customer_unique_id", "customer_city", "customer_state")
dim_seller = sellers.select("seller_id", "seller_city", "seller_state")
dim_product = products.select("product_id", "product_category_name", "product_weight_g")

In [ ]:
# dim_date: uma linha por dia, do min ao max das compras (grao = dia)
bounds = orders.select(
    F.min("order_purchase_timestamp").alias("mn"),
    F.max("order_purchase_timestamp").alias("mx"),
).first()

dim_date = (
    spark.sql(
        f"SELECT explode(sequence(to_date('{bounds.mn}'), to_date('{bounds.mx}'), interval 1 day)) AS date"
    )
    # chave inteira yyyymmdd (opcional, boa p/ joins e ordenacao)
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("year_month", F.date_format("date", "yyyy-MM"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("day_of_week", F.dayofweek("date"))            # 1=domingo
    .withColumn("weekday_name", F.date_format("date", "EEEE"))
)

In [ ]:
# Pagamentos agregados por pedido (payment_value -> alvo de OLS)
pay_agg = payments.groupBy("order_id").agg(F.sum("payment_value").alias("payment_value"))

# fact_orders (grao: pedido)
fact_orders = (
    orders.join(pay_agg, "order_id", "left")
    .withColumn("order_date", F.to_date("order_purchase_timestamp"))
)

# fact_order_items (grao: item)
fact_order_items = items.select(
    "order_id", "order_item_id", "product_id", "seller_id", "price", "freight_value"
)

In [ ]:
# Gravar Gold + OPTIMIZE (V-Order para Direct Lake)
gold = {
    "dim_customer": dim_customer,
    "dim_seller": dim_seller,
    "dim_product": dim_product,
    "dim_date": dim_date,
    "fact_orders": fact_orders,
    "fact_order_items": fact_order_items,
}
for name, d in gold.items():
    d.write.mode("overwrite").format("delta").saveAsTable(tbl(f"gold_{name}"))
    spark.sql(f"OPTIMIZE {tbl('gold_' + name)}")
    print("gravado gold_" + name)